# Predicting Residential EV Charging Loads using Neural Networks

Build a neural network in PyTorch that predicts the energy (kWh) drawn during EV charging
sessions at residential apartment buildings in Norway. You'll clean the data, fold in local
traffic counts as extra features, fit a linear-regression baseline, then train and compare a
neural network.

The data files live in `datasets/` (`EV charging reports.csv`, `Local traffic distribution.csv`)
and a longer-trained reference model is at `models/model4500.pth`.

### ⚠️ Heads-up on the raw data (read before starting)
These are the raw Norway/Mendeley files, so a few details differ from the clean Codecademy task
wording. They're called out again as hint toggles on each task — they're nudges, not full
solutions:

- **Both CSVs are semicolon-delimited** (`;`) and numbers use **commas as decimals** (`0,3`), so
  you must read with `pd.read_csv(..., sep=';')`.
- In the raw EV file `Start_plugin_hour` is just an **integer hour**, so it won't merge against
  the traffic `Date_from` timestamps. Build an hour-floored timestamp from `Start_plugin` first
  (Task 3).
- The file still carries non-numeric string columns (`User_type`, `month_plugin`,
  `weekdays_plugin`) and the traffic sensors use `'-'` for missing readings — handle these in the
  cleaning steps (Tasks 5–7) so `.astype(float)` succeeds.
- `models/model4500.pth` was trained on a **26-feature, one-hot-encoded** version of the data. If
  you drop the categorical columns you'll get fewer features and that saved model won't load
  against your tensors (Task 19).

In [ ]:
# Setup - import basic data libraries
import numpy as np
import pandas as pd

## Task Group 1 - Load, Inspect, and Merge Datasets

### Task 1
The file `datasets/EV charging reports.csv` contains electric vehicle (EV) charging data. These
come from various residential apartment buildings in Norway. The data includes specific user and
garage information, plug-in and plug-out times, charging loads, and the dates of charging sessions.

Import this CSV file to a pandas DataFrame named `ev_charging_reports`.

Use the `.head()` method to preview the first five rows.

In [6]:
ev_charging_reports = pd.read_csv("datasets/EV charging reports.csv", sep=';')
ev_charging_reports.head()

,session_ID,Garage_ID,User_ID,User_type,Shared_ID,Start_plugin,Start_plugin_hour,End_plugout,End_plugout_hour,El_kWh,Duration_hours,month_plugin,weekdays_plugin,Plugin_category,Duration_category
0,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3","0,05",Dec,Friday,late morning (9-12),Less than 3 hours
1,2,AdO3,AdO3-4,Private,NaN,21.12.2018 10:24,10,21.12.2018 10:32,10.0,"0,87","0,136666667",Dec,Friday,late morning (9-12),Less than 3 hours
2,3,AdO3,AdO3-4,Private,NaN,21.12.2018 11:33,11,21.12.2018 19:46,19.0,"29,87","8,216388889",Dec,Friday,late morning (9-12),Between 6 and 9 hours
3,4,AdO3,AdO3-2,Private,NaN,22.12.2018 16:15,16,23.12.2018 16:40,16.0,"15,56","24,41972222",Dec,Saturday,late afternoon (15-18),More than 18 hours
4,5,AdO3,AdO3-2,Private,NaN,24.12.2018 22:03,22,24.12.2018 23:02,23.0,"3,62","0,970555556",Dec,Monday,late evening (21-midnight),Less than 3 hours


<details><summary><b>What is the structure of the dataset?</b></summary>

This file is **semicolon-delimited**, so pass `sep=';'` to `pd.read_csv` — otherwise every row
collapses into a single column. Each row is one charging *session*: identifiers
(`session_ID`, `Garage_ID`, `User_ID`), plug-in / plug-out times and hours, the energy drawn
(`El_kWh`, the target), session `Duration_hours`, and categorical fields like `User_type`,
`month_plugin`, and `weekdays_plugin`.

</details>

### Task 2
Import the file `datasets/Local traffic distribution.csv` to a pandas DataFrame named
`traffic_reports`. This dataset contains the hourly local traffic-density counts at 5 nearby
traffic locations.

Preview the first five rows.

In [7]:
traffic_reports = pd.read_csv("datasets/Local traffic distribution.csv", sep=';')
traffic_reports.head()

,Date_from,Date_to,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,01.12.2018 00:00,01.12.2018 01:00,639,0,0,4,144
1,01.12.2018 01:00,01.12.2018 02:00,487,153,115,21,83
2,01.12.2018 02:00,01.12.2018 03:00,408,85,75,10,69
3,01.12.2018 03:00,01.12.2018 04:00,282,89,56,8,39
4,01.12.2018 04:00,01.12.2018 05:00,165,64,34,3,25


<details><summary><b>What is the structure of the dataset?</b></summary>

Also **semicolon-delimited**. Columns are `Date_from` / `Date_to` (the hour window) plus one count
column per sensor (`KROPPAN BRU`, `MOHOLTLIA`, `SELSBAKK`, `MOHOLT RAMPE 2`,
`Jonsvannsveien vest for Steinanvegen`). Missing readings appear as the string `'-'`, so these
columns load as `object`, not numbers.

</details>

### Task 3
We'd like to use the traffic data to help our model. The same charging location may charge at
different rates depending on the number of cars being charged, so this traffic data might help the
model out.

Merge the `ev_charging_reports` and `traffic_reports` datasets together into a DataFrame named
`ev_charging_traffic`, joining on:
- `Start_plugin_hour` in `ev_charging_reports`
- `Date_from` in `traffic_reports`

In [9]:
# This is the way to convert integer time to "10" to 07/09/1999 example.  
ev_charging_reports['Start_plugin_hour'] = (
    pd.to_datetime(ev_charging_reports['Start_plugin'], format='%d.%m.%Y %H:%M')
      .dt.floor('h').dt.strftime('%d.%m.%Y %H:%M')
)

# .merge() is way of combining panda dataframes and left_on and right_on the columns 
ev_charging_traffic = ev_charging_reports.merge(traffic_reports, left_on = "Start_plugin_hour", right_on = "Date_from")
ev_charging_traffic.head()

,session_ID,Garage_ID,User_ID,User_type,Shared_ID,Start_plugin,Start_plugin_hour,End_plugout,End_plugout_hour,El_kWh,...,weekdays_plugin,Plugin_category,Duration_category,Date_from,Date_to,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,21.12.2018 10:00,21.12.2018 10:23,10.0,"0,3",...,Friday,late morning (9-12),Less than 3 hours,21.12.2018 10:00,21.12.2018 11:00,3244,1632,545,194,622
1,2,AdO3,AdO3-4,Private,NaN,21.12.2018 10:24,21.12.2018 10:00,21.12.2018 10:32,10.0,"0,87",...,Friday,late morning (9-12),Less than 3 hours,21.12.2018 10:00,21.12.2018 11:00,3244,1632,545,194,622
2,3,AdO3,AdO3-4,Private,NaN,21.12.2018 11:33,21.12.2018 11:00,21.12.2018 19:46,19.0,"29,87",...,Friday,late morning (9-12),Between 6 and 9 hours,21.12.2018 11:00,21.12.2018 12:00,3605,1691,605,230,771
3,4,AdO3,AdO3-2,Private,NaN,22.12.2018 16:15,22.12.2018 16:00,23.12.2018 16:40,16.0,"15,56",...,Saturday,late afternoon (15-18),More than 18 hours,22.12.2018 16:00,22.12.2018 17:00,3052,1484,453,224,694
4,5,AdO3,AdO3-2,Private,NaN,24.12.2018 22:03,24.12.2018 22:00,24.12.2018 23:02,23.0,"3,62",...,Monday,late evening (21-midnight),Less than 3 hours,24.12.2018 22:00,24.12.2018 23:00,1390,693,226,83,353


<details><summary><b>Hint (raw data)</b></summary>

In the raw file `Start_plugin_hour` is just an **integer hour** (e.g. `10`), so it can't match the
`dd.mm.YYYY HH:00` timestamps in `Date_from`. First build an hour-floored timestamp string from
`Start_plugin`, e.g.

```python
ev_charging_reports['Start_plugin_hour'] = (
    pd.to_datetime(ev_charging_reports['Start_plugin'], format='%d.%m.%Y %H:%M')
      .dt.floor('h').dt.strftime('%d.%m.%Y %H:%M')
)
ev_charging_traffic = ev_charging_reports.merge(
    traffic_reports, left_on='Start_plugin_hour', right_on='Date_from')
```

</details>

### Task 4
Use `.info()` to inspect the merged dataset. Specifically, pay attention to the data types and the
number of missing values in each column.

In [10]:
ev_charging_traffic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6878 entries, 0 to 6877
Data columns (total 22 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   session_ID                            6878 non-null   int64  
 1   Garage_ID                             6878 non-null   object 
 2   User_ID                               6878 non-null   object 
 3   User_type                             6878 non-null   object 
 4   Shared_ID                             1412 non-null   object 
 5   Start_plugin                          6878 non-null   object 
 6   Start_plugin_hour                     6878 non-null   object 
 7   End_plugout                           6844 non-null   object 
 8   End_plugout_hour                      6844 non-null   float64
 9   El_kWh                                6878 non-null   object 
 10  Duration_hours                        6844 non-null   object 
 11  month_plugin     

<details><summary><b>What do we notice about the merged dataset under inspection?</b></summary>

`El_kWh`, `Duration_hours`, and the five traffic-sensor columns come in as `object` (string) dtype
rather than numeric — because of the comma decimals and the `'-'` missing markers. You'll convert
and clean these in Task Group 2 before any model can use them.

</details>

## Task Group 2 - Data Cleaning and Preparation

### Task 5
Let's start by reducing the size of our dataset by dropping columns that won't be used for
training. These include:
- ID columns
- columns with lots of missing data
- non-numeric columns (for now, since we haven't yet covered using non-numeric data in neural
  networks)

Drop the columns you don't want to use in training from `ev_charging_traffic`. To match the
solution, drop:

```python
['session_ID', 'Garage_ID', 'User_ID',
 'Shared_ID',
 'Plugin_category', 'Duration_category',
 'Start_plugin', 'Start_plugin_hour', 'End_plugout', 'End_plugout_hour',
 'Date_from', 'Date_to']
```

In [11]:
drop_columns = ['session_ID', 'Garage_ID', 'User_ID', 
                'Shared_ID',
                'Plugin_category','Duration_category', 
                'Start_plugin', 'Start_plugin_hour', 'End_plugout', 'End_plugout_hour', 
                'Date_from', 'Date_to']

ev_charging_traffic = ev_charging_traffic.drop(columns=drop_columns, axis=1)
ev_charging_traffic.head()

,User_type,El_kWh,Duration_hours,month_plugin,weekdays_plugin,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,Private,"0,3","0,05",Dec,Friday,3244,1632,545,194,622
1,Private,"0,87","0,136666667",Dec,Friday,3244,1632,545,194,622
2,Private,"29,87","8,216388889",Dec,Friday,3605,1691,605,230,771
3,Private,"15,56","24,41972222",Dec,Saturday,3052,1484,453,224,694
4,Private,"3,62","0,970555556",Dec,Monday,1390,693,226,83,353


<details><summary><b>Hint (raw data)</b></summary>

With the raw data there are also leftover string categoricals to drop: `User_type`,
`month_plugin`, and `weekdays_plugin`. (To *keep* them later, one-hot encode instead of dropping —
see Task 19.)

</details>

### Task 6
Earlier we saw that the `El_kWh` and `Duration_hours` columns were `object` data types. Upon
further inspection, we see the reason is that the data follows European notation, where commas `,`
are used as decimals instead of periods.

Replace `,` with `.` in the affected columns.

In [13]:
# Changing convention from (, to .). 
for column in ev_charging_traffic.columns:
    if ev_charging_traffic[column].dtype == "object":
         ev_charging_traffic[column] = ev_charging_traffic[column].str.replace(',', '.')
        
ev_charging_traffic.head()        

,User_type,El_kWh,Duration_hours,month_plugin,weekdays_plugin,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,Private,0.3,0.05,Dec,Friday,3244,1632,545,194,622
1,Private,0.87,0.136666667,Dec,Friday,3244,1632,545,194,622
2,Private,29.87,8.216388889,Dec,Friday,3605,1691,605,230,771
3,Private,15.56,24.41972222,Dec,Saturday,3052,1484,453,224,694
4,Private,3.62,0.970555556,Dec,Monday,1390,693,226,83,353


<details><summary><b>Hint (raw data)</b></summary>

The traffic-sensor columns are strings with commas too, and use `'-'` for missing readings.
Replace `'-'` with `NaN` (e.g. via `.replace('-', np.nan)`) so the rows can be dropped in Task 7,
and run the `,`→`.` replacement with `.str.replace(',', '.')` on every affected column.

</details>

---
## 📝 Review — Tasks 1–6 (load → merge → clean)

**The flow:** read 2 CSVs → make their time keys match → `merge` → inspect with `.info()` → drop junk columns → fix European decimals so columns can become numbers.

| # | Line of code | What it does |
|---|---|---|
| 1–2 | `pd.read_csv(path, sep=';')` | Load a CSV into a DataFrame. `sep=';'` because the file is **semicolon**-delimited (default is `,`) — without it every row mashes into one column. |
| 1–2 | `df.head()` | Preview the **first 5 rows** to sanity-check shape and columns. |
| 3 | `pd.to_datetime(col, format='%d.%m.%Y %H:%M')` | Parse a text column into real **datetime** objects, telling pandas the exact format. |
| 3 | `.dt.floor('h')` | Round each timestamp **down to the hour** (10:23 → 10:00). `.dt` = the datetime accessor. |
| 3 | `.dt.strftime('%d.%m.%Y %H:%M')` | Turn datetimes **back into strings** so they match `Date_from`'s text exactly for the join. |
| 3 | `df1.merge(df2, left_on='A', right_on='B')` | **Join** two DataFrames where `df1.A == df2.B` (like a SQL join). |
| 4 | `df.info()` | Summary: column names, **non-null counts** (→ missing data) and **dtypes** (`object` = string). |
| 5 | `df.drop(columns=cols)` | Return df **without** the listed columns (IDs, leftovers, categoricals). |
| 6 | `df[col].dtype == "object"` | Test a column's **type** — `object` means it's still text, not numeric. (`df[col]` alone is the whole Series — you need `.dtype`.) |
| 6 | `df[col].str.replace(',', '.')` | On a **string** column, swap European decimal comma → period so it can later cast to `float`. `.str` = the string accessor. |

### Reading a pandas DataFrame — quick reference
| You want | Code |
|---|---|
| One column (a Series) | `df['col']` |
| Several columns | `df[['a','b']]` |
| Row by position | `df.iloc[0]` |
| Row by label | `df.loc[idx]` |
| Shape `(rows, cols)` | `df.shape` |
| All column types | `df.dtypes` |
| Filter rows by condition | `df[df['col'] > 5]` |
| Element-wise on text | `df['col'].str.<method>()` |
| Element-wise on dates | `df['col'].dt.<method>()` |

**Gotcha I hit (Task 6):** `if df[col] == "object"` compares *every value* in the column → returns a Series → `if` can't collapse it to one True/False (`ValueError: truth value of a Series is ambiguous`). Test the **type** instead: `if df[col].dtype == "object":`. Rule of thumb — `df[col]` is the data, `df[col].dtype` is its type.

---

### Task 7
Next, convert the data types of all the columns of `ev_charging_traffic` to floats.

#### 🧩 Task 7 — code pieces & docs (assemble these yourself; not the final answer)

**Goal:** every column of `ev_charging_traffic` ends up as `float`.

| Piece | Code | What it does / docs |
|---|---|---|
| **A. Inspect** | `ev_charging_traffic.dtypes` | Returns each column's type. Use it to spot which columns are still `object` (text) and therefore can't become floats yet. |
| **B. Drop NaN rows** | `ev_charging_traffic.dropna()` | Returns a copy with any row containing a missing value removed. Needed because the `'-'` traffic readings became `NaN`. Default drops a row if **any** column is NaN (`how='any'`). |
| **C. Cast types** | `ev_charging_traffic.astype(float)` | Returns a copy with every column converted to `float`. Raises if a column holds non-numeric text. |
| **D. One-hot encode** *(optional)* | `pd.get_dummies(ev_charging_traffic, columns=['User_type','month_plugin','weekdays_plugin'])` | Replaces each text-category column with 0/1 indicator columns (e.g. `weekdays_plugin_Friday`). Use only if you want to **keep** the categoricals — it's also what the 26-feature `model4500.pth` needs later. |

**Docs to understand before you assemble them:**

- **They return a *new* frame — nothing changes in place.** Reassign the result:
  ```python
  ev_charging_traffic = ev_charging_traffic.dropna()
  ```
  Calling `ev_charging_traffic.dropna()` on its own just throws the result away.

- **`astype(float)` only works on numeric-looking data.** A column still holding text like `'Private'`, `'Dec'`, or `'Friday'` raises:
  `ValueError: could not convert string to float: 'Private'`.
  In your notebook those 3 categoricals are still present (you kept them in Task 5), so deal with them **before** casting.

- **Order matters:** handle the text categoricals (drop them, or Piece D) → `dropna` → `astype(float)`.

➡️ **Write your actual solution in the next cell.**

In [15]:
for column in ev_charging_traffic.columns:
    ev_charging_traffic[column] = ev_charging_traffic[column].dropna()
    ev_charging_traffic[column] = ev_charging_traffic[column].astype(float)
ev_charging_traffic.head()    

,User_type,El_kWh,Duration_hours,month_plugin,weekdays_plugin,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,Private,0.3,0.05,Dec,Friday,3244,1632,545,194,622
1,Private,0.87,0.136666667,Dec,Friday,3244,1632,545,194,622
2,Private,29.87,8.216388889,Dec,Friday,3605,1691,605,230,771
3,Private,15.56,24.41972222,Dec,Saturday,3052,1484,453,224,694
4,Private,3.62,0.970555556,Dec,Monday,1390,693,226,83,353


<details><summary><b>Hint (raw data)</b></summary>

Drop any rows still holding missing values first (`.dropna()`) so the cast to `float` succeeds, then `ev_charging_traffic = ev_charging_traffic.astype(float)`.

</details>

## Task Group 3 - Train Test Split
Next, let's split the dataset into training and testing datasets. The training data will be used to
train the model and the testing data will be used to evaluate the model.

### Task 8
First, create two datasets from `ev_charging_traffic`:
- `X` contains only the input numerical features
- `y` contains only the target column `El_kWh`

In [ ]:
# your code here


### Task 9
Use `sklearn` to split `X` and `y` into training and testing datasets. The training set should use
80% of the data. Set the `random_state` parameter to `2`.

In [ ]:
# your code here


## Task Group 4 - Linear Regression Baseline
This section is optional, but useful. The idea is to compare our neural network to a basic linear
regression. After all, if a basic linear regression works just as well, there's no need for the
neural network!

### Task 10
Use Scikit-learn to train a `LinearRegression` model using the training data to predict EV charging
loads. The linear regression will be used as a baseline to compare against the neural network we
will train later.

In [ ]:
# your code here


### Task 11
Evaluate the linear regression baseline by calculating the MSE on the testing data. Use
`mean_squared_error` from `sklearn.metrics`. Save the testing MSE to the variable `test_mse` and
print it out.

Codecademy's column set gives an MSE around `131.4` (about `11.5` kWh off on average once you take
the square root). With the raw features here you'll get a somewhat different number — that's
expected.

In [ ]:
# your code here


## Task Group 5 - Train a Neural Network Using PyTorch
Let's now create a neural network using PyTorch to predict EV charging loads.

### Task 12
First, we'll need to import the PyTorch library and modules.
- Import the PyTorch library `torch`.
- From `torch`, import `nn` to access built-in code for constructing networks and defining loss
  functions.
- From `torch`, import `optim` to access built-in optimizer algorithms.

In [ ]:
# your code here


### Task 13
Before training the neural network, convert the training and testing sets into PyTorch tensors and
specify `float` as the data type for the values.

*Hint: reshaping `y` to a column vector (`.view(-1, 1)`) makes its shape match the 1-node output.*

In [ ]:
# your code here


### Task 14
Next, let's use `nn.Sequential` to create a neural network.

First, set a random seed using `torch.manual_seed(42)`.

Then, create a sequential neural network with the following architecture:
- input layer with number of nodes equal to the number of training features
- a first hidden layer with `56` nodes and a ReLU activation
- a second hidden layer with `26` nodes and a ReLU activation
- an output layer with `1` node

Save the network to the variable `model`.

In [ ]:
# your code here


### Task 15
Next, let's define the loss function and optimizer used for training:
- set the MSE loss function to the variable `loss`
- set the Adam optimizer to the variable `optimizer` with a learning rate of `0.0007`

In [ ]:
# your code here


### Task 16
Create a training loop to train our neural network for 3000 epochs.

Keep track of the training loss by printing out the MSE every 500 epochs.

In [ ]:
# your code here


### Task 17
Save the neural network in the `models` directory using the path `models/model.pth`.

In [ ]:
# your code here


### Task 18
Evaluate the neural network on the testing set.

Save the testing data loss to the variable `test_loss` and use `.item()` to extract and print out
the loss.

*Hint: wrap the forward pass in `torch.no_grad()` so no gradients are tracked.*

In [ ]:
# your code here


### Task 19
We trained this same model for 4500 epochs locally. That model is saved as `models/model4500.pth`.
Load this model using PyTorch and evaluate it. How well does the longer-trained model perform?

In [ ]:
# your code here


<details><summary><b>Hint (raw data)</b></summary>

This saved model expects **26 input features** (it was trained on a one-hot-encoded version of the
data). If your pipeline dropped the categorical columns it won't load against your tensors —
compare `model4500[0].in_features` with `X_test_tensor.shape[1]`. To actually use it, one-hot
encode `month_plugin` / `weekdays_plugin` / `User_type` back in instead of dropping them.

</details>

## Wrap-up
That's the end of the project on predicting EV charging loads! Some things you might investigate
further:
- explore different ways to clean and prepare the data (e.g. one-hot encode the categoricals)
- more features don't guarantee a better model — test out different sets of input columns
- tune the number of nodes in the hidden layers, the activation functions, and the learning rate
- train for a larger number of epochs